# EE 467 – Phishing Detection: Model Training & Evaluation

**Approach 1:** Logistic Regression (baseline, interpretable)  
**Approach 2a:** Random Forest  
**Approach 2b:** Gradient Boosting


In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_loader import load_dataset, FEATURE_NAMES
from src.preprocessing import prepare_all
from src.models import (
    get_default_models, get_feature_importances,
    tune_model, LR_PARAM_GRID, RF_PARAM_GRID, GB_PARAM_GRID,
    build_logistic_regression, build_random_forest, build_gradient_boosting
)
from src.evaluation import (
    evaluate_all_models, plot_confusion_matrices, plot_roc_curves,
    plot_precision_recall_curves, plot_metric_comparison, plot_feature_importances
)

sns.set_theme(style='whitegrid')
SEED = 42

## 1. Load & Preprocess

In [ ]:
df = load_dataset()
data = prepare_all(df, random_state=SEED)

## 2. Baseline: Train with Default Hyperparameters

In [ ]:
models = get_default_models(random_state=SEED)

# Logistic Regression uses scaled features
models['Logistic Regression'].fit(data['X_train_sc'], data['y_train'])

# Tree models use raw features
models['Random Forest'].fit(data['X_train'], data['y_train'])
models['Gradient Boosting'].fit(data['X_train'], data['y_train'])

print('All models trained.')

## 3. Validation Set Evaluation

In [ ]:
val_summary = evaluate_all_models(models, data, split='val')
val_summary[['accuracy','precision','recall','f1','fpr','roc_auc']]

## 4. Hyperparameter Tuning (Optional — takes ~5 min)

Set `RUN_TUNING = True` to enable grid search.

In [ ]:
RUN_TUNING = False  # Set to True to run grid search

if RUN_TUNING:
    print('Tuning Logistic Regression...')
    lr_tuned, _, lr_best = tune_model(
        build_logistic_regression(random_state=SEED), LR_PARAM_GRID,
        data['X_train_sc'], data['y_train'], scoring='f1'
    )

    print('\nTuning Random Forest...')
    rf_tuned, _, rf_best = tune_model(
        build_random_forest(random_state=SEED), RF_PARAM_GRID,
        data['X_train'], data['y_train'], scoring='f1'
    )

    print('\nTuning Gradient Boosting...')
    gb_tuned, _, gb_best = tune_model(
        build_gradient_boosting(random_state=SEED), GB_PARAM_GRID,
        data['X_train'], data['y_train'], scoring='f1'
    )

    tuned_models = {
        'Logistic Regression': lr_tuned,
        'Random Forest': rf_tuned,
        'Gradient Boosting': gb_tuned,
    }
    print('\n── Tuned model validation performance ──')
    val_summary_tuned = evaluate_all_models(tuned_models, data, split='val')
    models = tuned_models  # Use tuned models going forward
else:
    print('Skipping tuning (RUN_TUNING=False). Using default models.')

## 5. Final Test Set Evaluation

In [ ]:
test_summary = evaluate_all_models(models, data, split='test')
test_summary[['accuracy','precision','recall','f1','fpr','roc_auc']]

## 6. Plots

In [ ]:
plot_confusion_matrices(models, data, split='test')

In [ ]:
plot_roc_curves(models, data, split='test')

In [ ]:
plot_precision_recall_curves(models, data, split='test')

In [ ]:
plot_metric_comparison(test_summary)

## 7. Feature Importances

In [ ]:
for name, model in models.items():
    try:
        imp_df = get_feature_importances(model, FEATURE_NAMES)
        plot_feature_importances(imp_df, name, top_n=15)
        print(f'\nTop 5 features for {name}:')
        print(imp_df.head(5).to_string(index=False))
    except ValueError as e:
        print(f'{name}: {e}')

## 8. Threshold Analysis

Explore the recall vs. FPR tradeoff by adjusting the decision threshold.

In [ ]:
from sklearn.metrics import recall_score, precision_score, f1_score

# Use best tree model (typically GB or RF)
model_name = 'Gradient Boosting'
model = models[model_name]
X_test = data['X_test']
y_test = data['y_test']

probs = model.predict_proba(X_test)
# prob of phishing (class -1)
phishing_idx = list(model.classes_).index(-1)
prob_phishing = probs[:, phishing_idx]

thresholds = np.linspace(0.1, 0.9, 50)
recalls, fprs, f1s, precisions = [], [], [], []

for t in thresholds:
    y_pred = np.where(prob_phishing >= t, -1, 1)
    from sklearn.metrics import confusion_matrix
    cm = confusion_matrix(y_test, y_pred, labels=[-1, 1])
    tn, fp, fn, tp = cm.ravel()
    recalls.append(tp / (tp + fn) if (tp + fn) > 0 else 0)
    fprs.append(fp / (fp + tn) if (fp + tn) > 0 else 0)
    f1s.append(f1_score(y_test, y_pred, pos_label=-1, zero_division=0))
    precisions.append(precision_score(y_test, y_pred, pos_label=-1, zero_division=0))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(thresholds, recalls, label='Recall (phishing caught)', color='#e74c3c')
axes[0].plot(thresholds, fprs, label='FPR (false alarms)', color='#3498db')
axes[0].plot(thresholds, f1s, label='F1 Score', color='#2ecc71')
axes[0].set_xlabel('Decision Threshold')
axes[0].set_ylabel('Score')
axes[0].set_title(f'{model_name}: Threshold vs. Metrics')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(fprs, recalls, color='purple')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('Recall')
axes[1].set_title(f'{model_name}: Recall vs. FPR Tradeoff')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()